# 8. Train Cell-Trajectory JEPA

Phase A of the JEPA research trajectory: train a latent src→tgt predictor on LPS pairs.

- No MaskGIT demask/remask loop
- Loss = MSE in embedding space
- Default encoder: **pretrained MaskGIT / scmaskgit** source encoder (`pretraining_cohort`)
- Alternative: lightweight `CellEncoder` via `--jepa_encoder cell` (src IDs remapped to HVG)
- Outputs on sod2; GPUs `0,1`
- Loss curves: CSV + TensorBoard under sod2 `logs/` (plus WandB offline)

### Recommended: screen

```bash
screen -S jepa_train
bash /home/stuke1/perturbgen/Perturbgen/docs/examples/run_train_jepa_sod2.sh
```

Watch live with TensorBoard (from another terminal):

```bash
tensorboard --logdir /mnt/sod2-project/csb4/stuke1/perturbgen/logs --bind_all
```


In [4]:
import os
from pathlib import Path

WORKSPACE = Path("/home/stuke1/perturbgen")
REPO = WORKSPACE / "Perturbgen"
TOKENIZED = WORKSPACE / "T_perturb" / "tokenized_data" / "LPS_all_tps_2k"
SOD2_ROOT = Path("/mnt/sod2-project/csb4/stuke1/perturbgen")

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DIR"] = str(SOD2_ROOT / "wandb")
os.environ["TMPDIR"] = str(SOD2_ROOT / "tmp")

OUTPUT_DIR = str(SOD2_ROOT / "T_perturb" / "res" / "jepa")
SRC_DATASET = str(TOKENIZED / "dataset_2000_hvg_src" / "normal.dataset")
TGT_DATASET_FOLDER = str(TOKENIZED / "dataset_2000_hvg_tgt")
SRC_ADATA = str(TOKENIZED / "h5ad_pairing_2000_hvg_src" / "normal.h5ad")
TGT_ADATA_FOLDER = str(TOKENIZED / "h5ad_pairing_2000_hvg_tgt")
MAPPING_DICT_PATH = str(TOKENIZED / "token_id_to_genename_2000_hvg.pkl")
TOKENID_TO_ROWID = str(TOKENIZED / "tokenid_to_rowid_2000_hvg.pkl")
# Pretrained cohort (not LPS-fine-tuned): warm-starts JEPA token_embedding via --ckpt_masking_path
PRETRAIN_CKPT = str(
    REPO / "pretraining_cohort"
    / "20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt"
)
print("OUTPUT_DIR", OUTPUT_DIR)
print("PRETRAIN_CKPT exists", Path(PRETRAIN_CKPT).is_file())


OUTPUT_DIR /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa
PRETRAIN_CKPT exists True


In [6]:
cmd = [
    "python", "-m", "perturbgen", "train-jepa",
    "--train_mode", "jepa",
    "--split", "True",
    "--splitting_mode", "stratified",
    "--split_obs", "cell_type_harmonized",
    "--output_dir", OUTPUT_DIR,
    "--log_dir", str(SOD2_ROOT / "logs"),
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--tokenid_to_rowid_path", TOKENID_TO_ROWID,
    "--encoder_path", PRETRAIN_CKPT,
    "--ckpt_masking_path", PRETRAIN_CKPT,
    "--jepa_encoder", "scmaskgit",  # or "cell" for lightweight CellEncoder
    "--freeze_jepa_encoder", "false",
    "--batch_size", "64",
    "--epochs", "20",
    "--cellgen_lr", "1e-4",
    "--cellgen_wd", "1e-4",
    "--num_layers", "2",
    "--d_ff", "1024",
    "--d_model", "768",
    "--pred_tps", "1", "2", "3",
    "--var_list", "cell_type_harmonized", "time_after_LPS",
    "--ema_decay", "0.996",
    "--normalize_latents", "true",
    "--jepa_loss", "mse",
    "--wandb_mode", "offline",
    "--seed", "0",
]
print(" ".join(cmd))


python -m perturbgen train-jepa --train_mode jepa --split False --output_dir /mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/jepa --log_dir /mnt/sod2-project/csb4/stuke1/perturbgen/logs --src_dataset /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --tokenid_to_rowid_path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/tokenid_to_rowid_2000_hvg.pkl --encoder_path /home/stuke1/perturbgen/Perturbgen/pretraining_cohort/20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch

## Plot loss curves

After (or during) training, CSV metrics land under sod2 `logs/<run_id>/version_*/metrics.csv`.

CLI (saves `docs/examples/jepa_curves.png`):

```bash
python /home/stuke1/perturbgen/Perturbgen/docs/examples/plot_jepa_curves.py
```

Or run the next cell to plot `train/jepa_loss` and `val/jepa_loss` here.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

LOG_DIR = Path("/mnt/sod2-project/csb4/stuke1/perturbgen/logs")
# Prefer newest metrics.csv; override RUN_GLOB if you want a specific run.
candidates = sorted(LOG_DIR.glob("*/version_*/metrics.csv"), key=lambda p: p.stat().st_mtime)
assert candidates, f"No metrics.csv under {LOG_DIR} yet — start training first."
metrics_path = candidates[-1]
print("Using", metrics_path)

df = pd.read_csv(metrics_path)
# Lightning: step+epoch logging → train/jepa_loss_epoch; val usually val/jepa_loss
plot_cols = [
    c
    for c in ("train/jepa_loss_epoch", "val/jepa_loss", "train/jepa_loss")
    if c in df.columns
]
assert plot_cols, f"No JEPA loss columns in {metrics_path}. Columns: {list(df.columns)}"

by_epoch = df.groupby("epoch", as_index=True)[plot_cols].last() if "epoch" in df.columns else df[plot_cols]

fig, ax = plt.subplots(figsize=(7, 4))
for c in plot_cols:
    s = by_epoch[c].dropna()
    if len(s):
        ax.plot(s.index, s.values, label=c, marker="o", ms=3)
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("JEPA loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Phases B–F

After `JEPATrainer.test` writes embeddings:

```bash
python -m perturbgen eval-jepa --phase all \
  --jepa_embeddings $OUTPUT_DIR/embeddings/jepa_cell_embeddings.pt \
  --output_dir $OUTPUT_DIR/eval
```

Phase D decoder training: `python -m perturbgen train-jepa-decoder ... --ckpt_jepa_path <jepa.ckpt>`
